In [1]:
!pip install alpaca-py torch scikit-learn pandas matplotlib

In [2]:
import os

def load_alpaca_keys(path="keys/alpaca_keys.txt"):
    keys = {}
    with open(path, "r") as f:
        for line in f:
            if "=" in line:
                k, v = line.strip().split("=", 1)
                keys[k] = v
    os.environ["APCA_API_KEY_ID"] = keys["APCA_API_KEY_ID"]
    os.environ["APCA_API_SECRET_KEY"] = keys["APCA_API_SECRET_KEY"]
    return keys

keys = load_alpaca_keys()
print("Loaded Alpaca keys:", keys.keys())

Loaded Alpaca keys: dict_keys(['APCA_API_KEY_ID', 'APCA_API_SECRET_KEY'])


In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import datetime as dt
import pytz

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

TICKERS = ["AAPL", "MSFT", "AMZN", "GOOG"] 
SEQ_LEN = 84
PATCH_LEN = 7

os.makedirs("models", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

Using device: cpu


In [4]:
def to_series(x):
    """Convert any input to a proper 1D pandas Series."""
    if isinstance(x, pd.DataFrame):
        return x.iloc[:, 0]
    return pd.Series(x)

def normalize(df):
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        df[col] = to_series(df[col]).astype(float)
    return df

def SMA(series, n):
    s = to_series(series).astype(float)
    return s.rolling(n).mean()

def EMA(series, n):
    s = to_series(series).astype(float)
    return s.ewm(span=n, adjust=False).mean()

def RSI(series, n=14):
    s = to_series(series).astype(float)
    delta = s.diff()
    up = delta.clip(lower=0).rolling(n).mean()
    down = -delta.clip(upper=0).rolling(n).mean()
    rs = up / (down + 1e-9)
    return 100 - (100 / (1 + rs))

def ATR(df, n=14):
    high = to_series(df["High"]).astype(float)
    low = to_series(df["Low"]).astype(float)
    close = to_series(df["Close"]).astype(float)

    hl = high - low
    hc = (high - close.shift()).abs()
    lc = (low - close.shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)

    return tr.rolling(n).mean()

def add_indicators(df):
    df = normalize(df.copy())

    df["returns"] = df["Close"].pct_change()
    df["sma10"] = SMA(df["Close"], 10)
    df["sma50"] = SMA(df["Close"], 50)
    df["ema12"] = EMA(df["Close"], 12)
    df["ema26"] = EMA(df["Close"], 26)
    df["rsi14"] = RSI(df["Close"], 14)
    df["atr14"] = ATR(df, 14)
    df["vol20"] = df["returns"].rolling(20).std()
    df["close_sma10_ratio"] = df["Close"] / df["sma10"] - 1
    df["close_sma50_ratio"] = df["Close"] / df["sma50"] - 1

    return df.fillna(0)

In [5]:
ALPACA_KEY = os.getenv("APCA_API_KEY_ID")
ALPACA_SECRET = os.getenv("APCA_API_SECRET_KEY")

if not ALPACA_KEY or not ALPACA_SECRET:
    raise ValueError("Alpaca API keys not found. Make sure they are loaded from your keys folder.")

# Create Alpaca data client
data_client = StockHistoricalDataClient(ALPACA_KEY, ALPACA_SECRET)

def download_ticker(ticker):
    """Download daily OHLCV data for one ticker from Alpaca using FREE IEX feed."""

    end = dt.datetime.now(pytz.UTC)
    start = end - dt.timedelta(days=365*10)   # 10 years of data

    req = StockBarsRequest(
        symbol_or_symbols=ticker,
        timeframe=TimeFrame.Day,
        start=start,
        end=end,
        feed="iex"
    )

    # Fetch data
    bars = data_client.get_stock_bars(req).df

    if isinstance(bars.index, pd.MultiIndex):
        bars = bars.xs(ticker)

    df = bars.rename(columns={
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "volume": "Volume"
    })

    df.index = pd.to_datetime(df.index)

    df = df[["Open", "High", "Low", "Close", "Volume"]]

    return df


def download_all():
    """Download and combine all tickers into one ML dataset."""

    print("Starting full dataset download from Alpaca (IEX feed)...")
    dfs = []

    for t in TICKERS:
        print(f"→ Downloading {t}...")
        df = download_ticker(t)

        df = add_indicators(df)

        df["ticker"] = t
        dfs.append(df)

    full = pd.concat(dfs).sort_index()
    print("Done! Combined dataset shape:", full.shape)
    return full

df = download_all()
df.head()

Starting full dataset download from Alpaca (IEX feed)...
→ Downloading AAPL...
→ Downloading MSFT...
→ Downloading AMZN...
→ Downloading GOOG...
Done! Combined dataset shape: (5408, 16)


,Open,High,Low,Close,Volume,returns,sma10,sma50,ema12,ema26,rsi14,atr14,vol20,close_sma10_ratio,close_sma50_ratio,ticker
timestamp,,,,,,,,,,,,,,,,
2020-07-27 04:00:00+00:00,374.955,379.510,373.935,379.440,201123.0,0.000000,0.0,0.0,379.440000,379.440000,0.0,0.0,0.0,0.0,0.0,AAPL
2020-07-27 04:00:00+00:00,3058.870,3090.880,3016.170,3055.600,47612.0,0.000000,0.0,0.0,3055.600000,3055.600000,0.0,0.0,0.0,0.0,0.0,AMZN
2020-07-27 04:00:00+00:00,201.455,203.945,200.940,203.840,323083.0,0.000000,0.0,0.0,203.840000,203.840000,0.0,0.0,0.0,0.0,0.0,MSFT
2020-07-27 04:00:00+00:00,1517.695,1540.040,1515.950,1530.165,40830.0,0.000000,0.0,0.0,1530.165000,1530.165000,0.0,0.0,0.0,0.0,0.0,GOOG
2020-07-28 04:00:00+00:00,377.190,378.050,373.155,373.190,151665.0,-0.016472,0.0,0.0,378.478462,378.977037,0.0,0.0,0.0,0.0,0.0,AAPL


In [6]:
FEATURE_COLS = [
    "Close","Volume","returns","sma10","sma50",
    "ema12","ema26","rsi14","atr14","vol20",
    "close_sma10_ratio","close_sma50_ratio"
]
df = df.dropna().copy()

scaler = StandardScaler()
scaler.fit(df[FEATURE_COLS].values)

print("Scaler trained.")
print("Number of rows used:", len(df))
print("Number of features:", len(FEATURE_COLS))

Scaler trained.
Number of rows used: 5408
Number of features: 12


In [7]:
class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, df, feature_cols, seq_len):
        """
        df: full dataframe (multiple tickers)
        feature_cols: input features
        seq_len: length of historical window
        """

        X = scaler.transform(df[feature_cols].values)
        y = df["returns"].values

        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32)
        self.seq_len = seq_len

        self.indices = [
            i for i in range(len(df) - seq_len - 1)
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        end = start + self.seq_len

        seq_x = self.X[start:end]  
        target = self.y[end]       

        return (
            torch.tensor(seq_x, dtype=torch.float32),
            torch.tensor([target], dtype=torch.float32)
        )


# Create dataset
dataset = SeqDataset(df, FEATURE_COLS, SEQ_LEN)

print("Dataset size:", len(dataset))
print("Example input shape:", dataset[0][0].shape)  
print("Example target:", dataset[0][1])

Dataset size: 5323
Example input shape: torch.Size([84, 12])
Example target: tensor([-0.0068])


In [8]:
train_size = int(len(dataset) * 0.80)
val_size = int(len(dataset) * 0.10)
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

# DataLoaders
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=64)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=64)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print("Test size:", len(test_ds))

xb, yb = next(iter(train_loader))
print("Batch X shape:", xb.shape) 
print("Batch y shape:", yb.shape)

Train size: 4258
Val size: 532
Test size: 533
Batch X shape: torch.Size([64, 84, 12])
Batch y shape: torch.Size([64, 1])


In [9]:
class PatchEmbedding(nn.Module):
    def __init__(self, seq_len, n_features, patch_len, model_dim=128):
        super().__init__()

        assert seq_len % patch_len == 0, "SEQ_LEN must be divisible by PATCH_LEN"

        self.patch_len = patch_len
        self.n_patches = seq_len // patch_len

        self.input_dim = patch_len * n_features

        self.proj = nn.Linear(self.input_dim, model_dim)

    def forward(self, x):
        B, L, F = x.shape

        patches = x.unfold(dimension=1, size=self.patch_len, step=self.patch_len)

        B, P, PL, F = patches.shape

        patches = patches.reshape(B, P, PL * F)

        return self.proj(patches) 


class PatchTST(nn.Module):
    def __init__(self, seq_len, n_features, patch_len, model_dim=128, n_heads=8, n_layers=4):
        super().__init__()

        self.patch_embed = PatchEmbedding(seq_len, n_features, patch_len, model_dim)

        n_patches = seq_len // patch_len


        self.pos_emb = nn.Parameter(torch.randn(1, n_patches, model_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=n_heads,
            dim_feedforward=model_dim * 4,
            dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.patch_embed(x) + self.pos_emb

        x = self.encoder(x) 

        x = x.transpose(1, 2) 
        x = self.pool(x).squeeze(-1) 

        return self.fc(x) 

In [10]:
model = PatchTST(
    seq_len=SEQ_LEN,
    n_features=len(FEATURE_COLS),
    patch_len=PATCH_LEN,
    model_dim=128,
    n_heads=8,
    n_layers=4
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

def train_epoch():
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(xb)

    return total_loss / len(train_loader.dataset)


def validate():
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            total_loss += loss.item() * len(xb)

    return total_loss / len(val_loader.dataset)


EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch()
    val_loss = validate()

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.6f} | "
          f"Val Loss: {val_loss:.6f}")

Epoch 01 | Train Loss: 0.007790 | Val Loss: 0.001068
Epoch 02 | Train Loss: 0.002422 | Val Loss: 0.000675
Epoch 03 | Train Loss: 0.002029 | Val Loss: 0.001918
Epoch 04 | Train Loss: 0.001804 | Val Loss: 0.000674
Epoch 05 | Train Loss: 0.001562 | Val Loss: 0.000456
Epoch 06 | Train Loss: 0.001480 | Val Loss: 0.000428
Epoch 07 | Train Loss: 0.001499 | Val Loss: 0.000931
Epoch 08 | Train Loss: 0.001426 | Val Loss: 0.000390
Epoch 09 | Train Loss: 0.001103 | Val Loss: 0.000972
Epoch 10 | Train Loss: 0.001119 | Val Loss: 0.000356


In [11]:
save_path = "models/patchtst_final.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "scaler": scaler,
    "feature_cols": FEATURE_COLS,
    "config": {
        "seq_len": SEQ_LEN,
        "patch_len": PATCH_LEN,
        "n_features": len(FEATURE_COLS)
    }
}, save_path)

print("Model saved successfully!")
print("Path:", save_path)

Model saved successfully!
Path: models/patchtst_final.pth
